# The Hinode (Solar-B) Mission: An Overview — Implementation / 구현

**Paper**: Kosugi, T., et al., *The Hinode (Solar-B) Mission: An Overview*, Solar Physics **243**, 3–17 (2007). DOI: 10.1007/s11207-007-9014-6

This notebook implements three numerical exercises tied to the paper:
1. **Instrument synergy table** — SOT / XRT / EIS coverage of photosphere → chromosphere → transition region → corona, including wavelengths, temperatures, FOV, cadence.
2. **Sun-synchronous orbit calculator** — derive the Hinode inclination from the J₂ secular precession formula and visualize the orbit-plane geometry that yields 9 months of continuous solar viewing.
3. **Photosphere–chromosphere–corona simultaneous diagnostic** — build a synthetic temperature–height profile of the solar atmosphere and overlay each Hinode instrument's diagnostic temperature window to demonstrate the multi-layer coverage.

이 노트북은 논문과 직접 연결되는 세 가지 수치 실습을 구현한다:
1. **기기 시너지 표** — SOT / XRT / EIS의 광구–채층–전이영역–코로나 진단 범위(파장, 온도, FOV, 케이던스).
2. **태양동기궤도 계산기** — J₂ 세속 진동 공식으로 Hinode 경사각을 도출, 9개월 연속 관측을 가능하게 하는 궤도면 기하 시각화.
3. **광구–채층–코로나 동시 진단** — 태양 대기 온도–고도 프로파일을 합성하고 각 Hinode 기기의 진단 온도 구간을 중첩 표시.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## Part 1: Hinode Instrument Synergy Table / 기기 시너지 표

We construct a structured DataFrame that captures the key Table 2 specifications from Kosugi et al. (2007) and then compute derived quantities such as diffraction limit, photons-per-pixel-per-second proxies, and FOV in arcsec². This view makes the **complementary roles** of SOT / EIS / XRT explicit: SOT measures the magnetic boundary condition, EIS measures the upper transition region and lower corona via Doppler-resolved EUV spectra, and XRT measures the hot corona over a wide FOV.

Kosugi 외(2007) Table 2의 핵심 사양을 구조화된 DataFrame으로 만들고 회절 한계, 단위 픽셀·초당 광자 프록시, FOV(arcsec²) 등을 계산한다. 이를 통해 SOT / EIS / XRT가 어떻게 **상보적 역할**을 분담하는지 — SOT는 자기 경계조건, EIS는 도플러 분해 EUV 분광으로 상부 전이영역·하부 코로나, XRT는 광시야로 고온 코로나 — 시각화한다.

In [ ]:
def build_instrument_table() -> pd.DataFrame:
    """Build the Hinode instrument synergy table.

    Returns:
        DataFrame with one row per instrument and columns capturing
        wavelength range, target plasma temperature, FOV, pixel size,
        time cadence, and diagnostic role per Kosugi et al. (2007) Table 2.
    """
    rows = [
        {
            'instrument': 'SOT/NFI',
            'aperture_cm': 50.0,
            'wavelength_min_A': 5170.0,
            'wavelength_max_A': 6700.0,
            'log_T_min': 3.7,  # photosphere ~5000 K
            'log_T_max': 4.3,  # chromosphere ~20000 K
            'fov_ew_arcsec': 328.0,
            'fov_ns_arcsec': 164.0,
            'pixel_arcsec': 0.08,
            'cadence_typical_s': 30.0,
            'role': 'Photospheric vector B (Fe I 6302/6301), Dopplergrams, Halpha',
        },
        {
            'instrument': 'SOT/BFI',
            'aperture_cm': 50.0,
            'wavelength_min_A': 3883.0,
            'wavelength_max_A': 6684.0,
            'log_T_min': 3.7,
            'log_T_max': 4.0,
            'fov_ew_arcsec': 218.0,
            'fov_ns_arcsec': 109.0,
            'pixel_arcsec': 0.053,
            'cadence_typical_s': 30.0,
            'role': 'Broadband continuum + Ca II H chromospheric proxy',
        },
        {
            'instrument': 'SOT/SP',
            'aperture_cm': 50.0,
            'wavelength_min_A': 6301.5,
            'wavelength_max_A': 6302.5,
            'log_T_min': 3.7,
            'log_T_max': 3.8,
            'fov_ew_arcsec': 320.0,
            'fov_ns_arcsec': 164.0,
            'pixel_arcsec': 0.16,
            'cadence_typical_s': 3600.0,  # ~1 hr full Stokes raster
            'role': 'Full Stokes IQUV at Fe I 6302/6301 (vector magnetography)',
        },
        {
            'instrument': 'EIS',
            'aperture_cm': 15.0,
            'wavelength_min_A': 170.0,
            'wavelength_max_A': 290.0,
            'log_T_min': 5.0,  # 100,000 K
            'log_T_max': 7.0,  # 10 MK
            'fov_ew_arcsec': 590.0,
            'fov_ns_arcsec': 512.0,
            'pixel_arcsec': 1.0,
            'cadence_typical_s': 10.0,  # active region
            'role': 'Upper TR / lower corona Doppler (3 km/s) and density',
        },
        {
            'instrument': 'XRT',
            'aperture_cm': 30.0,
            'wavelength_min_A': 2.0,
            'wavelength_max_A': 200.0,
            'log_T_min': 6.0,  # 1 MK
            'log_T_max': 7.5,  # 30 MK flares
            'fov_ew_arcsec': 2048.0,  # 34 arcmin
            'fov_ns_arcsec': 2048.0,
            'pixel_arcsec': 1.0,
            'cadence_typical_s': 4.0,  # nominal
            'role': 'Coronal X-ray imaging, 9 filters, dlog T = 0.2',
        },
    ]
    return pd.DataFrame(rows)


def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add derived diagnostic columns: diffraction limit and FOV area.

    Args:
        df: instrument table from build_instrument_table().

    Returns:
        Same DataFrame with new columns:
          - lambda_mid_A: midpoint wavelength (Angstrom).
          - diff_limit_arcsec: 1.22 * lambda / D in arcsec.
          - fov_arcmin2: field of view area in arcmin^2.
          - dT_decades: log-T diagnostic span.
    """
    out = df.copy()
    out['lambda_mid_A'] = 0.5 * (out['wavelength_min_A'] + out['wavelength_max_A'])
    # 1.22 * lambda / D, with lambda in m and D in m, gives radians; convert to arcsec.
    rad_to_arcsec = 206264.806
    lambda_m = out['lambda_mid_A'] * 1.0e-10
    aperture_m = out['aperture_cm'] * 1.0e-2
    out['diff_limit_arcsec'] = 1.22 * lambda_m / aperture_m * rad_to_arcsec
    out['fov_arcmin2'] = (out['fov_ew_arcsec'] * out['fov_ns_arcsec']) / 3600.0
    out['dT_decades'] = out['log_T_max'] - out['log_T_min']
    return out


instruments = add_derived_columns(build_instrument_table())
display_cols = [
    'instrument', 'lambda_mid_A', 'diff_limit_arcsec', 'pixel_arcsec',
    'fov_arcmin2', 'log_T_min', 'log_T_max', 'cadence_typical_s', 'role',
]
instruments[display_cols].round(3)

### Visualization: each instrument's diagnostic temperature window / 각 기기의 진단 온도 구간

We plot horizontal bars whose horizontal extent is each instrument's log T diagnostic range. The combined coverage shows that SOT, EIS, and XRT together span **logT ≈ 3.7 to 7.5** — the entire photosphere → flare-corona temperature range — without significant gaps. This is the central scientific case of the paper.

각 기기의 logT 진단 범위를 수평 막대로 시각화. SOT, EIS, XRT 결합 시 **logT ≈ 3.7 ~ 7.5** — 광구에서 플레어 코로나에 이르는 전 온도 범위 — 가 거의 빈틈 없이 덮인다는 것을 확인.

In [ ]:
def plot_temperature_coverage(df: pd.DataFrame) -> None:
    """Plot diagnostic temperature coverage as horizontal bars.

    Args:
        df: instruments DataFrame with log_T_min and log_T_max columns.
    """
    fig, ax = plt.subplots(figsize=(11, 5))
    color_map = {
        'SOT/NFI': '#1f77b4',
        'SOT/BFI': '#1aa3ff',
        'SOT/SP': '#003c7a',
        'EIS': '#2ca02c',
        'XRT': '#d62728',
    }
    y_positions = np.arange(len(df))[::-1]
    for y, (_, row) in zip(y_positions, df.iterrows()):
        ax.barh(
            y,
            row['log_T_max'] - row['log_T_min'],
            left=row['log_T_min'],
            color=color_map.get(row['instrument'], 'gray'),
            edgecolor='black',
            height=0.6,
        )
        ax.text(
            (row['log_T_min'] + row['log_T_max']) / 2,
            y,
            row['instrument'],
            ha='center',
            va='center',
            color='white',
            fontweight='bold',
        )
    # Reference temperature regions of the solar atmosphere
    regions = [
        (3.7, 3.8, 'photosphere\n5000 K'),
        (3.8, 4.3, 'chromosphere\n10^4 K'),
        (4.3, 6.0, 'transition\nregion'),
        (6.0, 6.7, 'corona\n1-5 MK'),
        (6.7, 7.5, 'flare\ncorona\n>10 MK'),
    ]
    for x0, x1, label in regions:
        ax.axvspan(x0, x1, alpha=0.05, color='orange')
        ax.text((x0 + x1) / 2, len(df) - 0.3, label, ha='center', va='bottom',
                fontsize=9, color='dimgray')
    ax.set_xlabel('log T  [K]')
    ax.set_ylabel('instrument')
    ax.set_yticks([])
    ax.set_title('Hinode instruments — diagnostic temperature coverage / 진단 온도 범위')
    ax.set_xlim(3.5, 7.7)
    ax.set_ylim(-0.7, len(df) + 0.3)
    plt.tight_layout()
    plt.show()


plot_temperature_coverage(instruments)

## Part 2: Sun-synchronous orbit calculator / 태양동기궤도 계산기

We derive the orbital inclination required to make the right ascension of ascending node (RAAN) precess at the Earth's mean motion around the Sun, using the J₂-only secular precession formula:
$$ \dot{\Omega} = -\frac{3}{2} \, J_2 \left(\frac{R_\oplus}{a}\right)^2 \, n \, \cos i \;=\; n_\odot. $$
We solve for `i` given Hinode's altitude h = 680 km and verify that `i ≈ 98.1°`.

지구 중력장의 J₂만 고려한 RAAN의 세속 진동률이 지구의 태양 둘레 평균 운동량과 같아져야 태양동기궤도가 된다. 식을 풀어 Hinode 고도 h = 680 km에서 경사각 i를 구하고 약 **98.1°**임을 검증한다.

In [ ]:
def sun_synchronous_inclination(altitude_km: float) -> float:
    """Compute inclination required for a Sun-synchronous orbit.

    Solves the J2-only secular precession condition:
        Omega_dot = - 1.5 * J2 * (R_E / a)^2 * n * cos(i) = n_sun,
    where n_sun = 2*pi / (365.25 * 86400) rad/s.

    Args:
        altitude_km: orbit altitude above Earth's surface in km.

    Returns:
        Required inclination in degrees (typically slightly above 90 deg,
        i.e. retrograde).
    """
    # Constants
    j2 = 1.0826e-3  # Earth oblateness coefficient
    r_earth_km = 6378.137  # equatorial radius (km)
    mu_earth_km3_s2 = 398600.4418  # gravitational parameter (km^3 / s^2)
    seconds_per_day = 86400.0
    days_per_year = 365.25
    n_sun = 2.0 * np.pi / (days_per_year * seconds_per_day)  # rad/s

    a_km = r_earth_km + altitude_km  # semi-major axis
    n = np.sqrt(mu_earth_km3_s2 / a_km**3)  # mean motion (rad/s)

    cos_i = -n_sun / (1.5 * j2 * (r_earth_km / a_km) ** 2 * n)
    inclination_rad = np.arccos(cos_i)
    return np.degrees(inclination_rad)


def orbital_period_minutes(altitude_km: float) -> float:
    """Return Keplerian orbital period in minutes for a circular orbit."""
    mu_earth_km3_s2 = 398600.4418
    r_earth_km = 6378.137
    a_km = r_earth_km + altitude_km
    period_s = 2.0 * np.pi * np.sqrt(a_km**3 / mu_earth_km3_s2)
    return period_s / 60.0


# Verify for Hinode: altitude 680 km --> i ~ 98.1, period ~ 98 min
altitude_hinode_km = 680.0
i_hinode = sun_synchronous_inclination(altitude_hinode_km)
period_hinode = orbital_period_minutes(altitude_hinode_km)
print(f'Hinode altitude: {altitude_hinode_km:.1f} km')
print(f'Required inclination for Sun-synchronous orbit: {i_hinode:.2f} deg')
print(f'Orbital period: {period_hinode:.2f} min')
print('Paper values: i = 98.1 deg, period = 98 min')

In [ ]:
# Sweep altitudes and visualize the inclination/period relationship
altitudes = np.linspace(300.0, 1500.0, 200)
inclinations = np.array([sun_synchronous_inclination(h) for h in altitudes])
periods = np.array([orbital_period_minutes(h) for h in altitudes])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(altitudes, inclinations, color='royalblue')
axes[0].axvline(680.0, color='red', linestyle='--', label='Hinode (680 km)')
axes[0].axhline(98.1, color='red', linestyle=':', alpha=0.5)
axes[0].set_xlabel('altitude  [km]')
axes[0].set_ylabel('Sun-synchronous inclination  [deg]')
axes[0].set_title('Inclination vs. altitude (J2 model) / 경사각 vs. 고도')
axes[0].legend()

axes[1].plot(altitudes, periods, color='seagreen')
axes[1].axvline(680.0, color='red', linestyle='--', label='Hinode (680 km)')
axes[1].axhline(98.0, color='red', linestyle=':', alpha=0.5)
axes[1].set_xlabel('altitude  [km]')
axes[1].set_ylabel('orbital period  [min]')
axes[1].set_title('Period vs. altitude / 주기 vs. 고도')
axes[1].legend()

plt.tight_layout()
plt.show()

### Eclipse-season geometry / 일식 계절 기하

A Sun-synchronous dawn–dusk orbit has its plane perpendicular to the Sun–Earth line at the equinoxes and oblique at the solstices. The orbit enters Earth's shadow only when the Sun's declination is small enough that the shadow cone intersects the orbit — this happens in a roughly 3-month window per year. We compute the maximum off-pointing angle β between the Sun and the orbit plane normal and identify the eclipse season as the interval where β falls below a threshold.

태양동기 dawn–dusk 궤도면은 분점에 태양–지구 선과 수직이고 지점에 비스듬해진다. 태양 적위가 충분히 작아 그림자 원뿔이 궤도와 교차하는 약 3개월 동안만 일식이 발생한다. 태양과 궤도면 법선 사이 각도 β를 계산하여 일식 계절을 식별한다.

In [ ]:
def sun_orbit_beta_angle(day_of_year: np.ndarray, inclination_deg: float = 98.1) -> np.ndarray:
    """Approximate the Sun-to-orbit-plane angle (beta) over a year.

    Approximation: the Sun's declination follows delta = 23.44 * sin(2*pi * (d - 80)/365).
    For a dawn-dusk Sun-synchronous orbit, the orbit-plane normal is approximately
    aligned with the Sun direction at the equinoxes; the angle between Sun and
    plane normal therefore tracks the solar declination modulated by orbit
    inclination.

    Args:
        day_of_year: array of days in [1, 365].
        inclination_deg: orbital inclination in degrees.

    Returns:
        Beta angle in degrees, defined here as 90 - delta_sun for a dawn-dusk
        orbit so that 90 deg corresponds to perfect alignment (no eclipse) and
        smaller values correspond to eclipse season.
    """
    declination_deg = 23.44 * np.sin(2.0 * np.pi * (day_of_year - 80.0) / 365.25)
    # Beta is the complement of declination for a dawn-dusk SSO
    beta_deg = 90.0 - np.abs(declination_deg)
    return beta_deg


def eclipse_threshold_deg(altitude_km: float) -> float:
    """Return the beta-angle threshold below which Earth's shadow eclipses the orbit.

    Geometry: orbit eclipsed when beta < arcsin(R_E / (R_E + h)).
    """
    r_earth_km = 6378.137
    a_km = r_earth_km + altitude_km
    return np.degrees(np.arcsin(r_earth_km / a_km))


doy = np.arange(1, 366)
beta = sun_orbit_beta_angle(doy, inclination_deg=98.1)
threshold = 90.0 - eclipse_threshold_deg(680.0)

in_eclipse = beta < threshold
n_eclipse_days = int(np.sum(in_eclipse))
n_full_sun_days = int(365 - n_eclipse_days)
print(f'Eclipse-season threshold beta: {threshold:.2f} deg')
print(f'Approximate days/year in eclipse season: {n_eclipse_days}')
print(f'Approximate days/year of continuous solar viewing: {n_full_sun_days}')
print('Paper claim: 9 months (~270 days) of continuous solar observation')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(doy, beta, color='gold', lw=2, label=r'$\beta$ (Sun vs. orbit plane normal)')
ax.axhline(threshold, color='red', linestyle='--', label='eclipse threshold')
ax.fill_between(doy, beta, threshold, where=in_eclipse, alpha=0.3, color='dimgray', label='eclipse season')
ax.set_xlabel('day of year')
ax.set_ylabel(r'$\beta$  [deg]')
ax.set_title('Hinode Sun-synchronous orbit eclipse season / Hinode 태양동기궤도 일식 계절')
ax.set_xlim(1, 365)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Part 3: Simultaneous photosphere–chromosphere–corona diagnostic / 광구–채층–코로나 동시 진단

We synthesize a simplified VAL-C-like temperature-height profile of the quiet-Sun atmosphere and overlay each Hinode instrument's diagnostic temperature window. This **vertical-stack picture** is the qualitative justification for placing SOT, EIS, and XRT on a single satellite with co-aligned pointing.

VAL-C 유사 정온 태양 대기 온도–고도 프로파일을 합성하고 각 Hinode 기기의 진단 온도 구간을 중첩 표시한다. 이 **수직 스택 이미지**가 SOT, EIS, XRT를 단일 위성·동일 지향으로 묶은 설계의 정성적 정당화다.

In [ ]:
def synthesize_quiet_sun_atmosphere(height_km: np.ndarray) -> np.ndarray:
    """Synthesize a simplified quiet-Sun temperature-height profile.

    Builds a piecewise approximation broadly resembling the VAL-C model
    plus a transition region jump and a coronal plateau:
        - photosphere: ~5800 K declining to ~4400 K at temperature minimum
        - chromosphere: gradual rise to ~10^4 K
        - transition region: sharp jump from 10^4 to 10^6 K
        - corona: plateau at ~1.5 MK

    Args:
        height_km: array of heights above tau=1 (km), positive upward.

    Returns:
        Temperature in K at each height. Shape matches input.
    """
    temperature = np.empty_like(height_km, dtype=float)
    # Photosphere: 0 - 500 km, T drops from 5800 to 4400 K
    photo = height_km < 500.0
    temperature[photo] = 5800.0 - (5800.0 - 4400.0) * (height_km[photo] / 500.0)
    # Chromosphere: 500 - 2000 km, T rises from 4400 to 10000 K
    chromo = (height_km >= 500.0) & (height_km < 2000.0)
    temperature[chromo] = 4400.0 + (10000.0 - 4400.0) * (height_km[chromo] - 500.0) / 1500.0
    # Transition region: 2000 - 2200 km, exponential jump 10^4 -> 10^6 K
    tr = (height_km >= 2000.0) & (height_km < 2200.0)
    log_t_tr = 4.0 + (6.0 - 4.0) * (height_km[tr] - 2000.0) / 200.0
    temperature[tr] = 10.0**log_t_tr
    # Corona: > 2200 km, ~1.5 MK plateau with mild rise
    corona = height_km >= 2200.0
    temperature[corona] = 1.5e6 + 5.0e5 * (1.0 - np.exp(-(height_km[corona] - 2200.0) / 5000.0))
    return temperature


heights_km = np.linspace(0.0, 20000.0, 4000)
temp_K = synthesize_quiet_sun_atmosphere(heights_km)
log_temp = np.log10(temp_K)

In [ ]:
def plot_atmosphere_with_instrument_windows(
    heights_km: np.ndarray,
    log_temp: np.ndarray,
    df: pd.DataFrame,
) -> None:
    """Plot the synthetic atmosphere with overlaid instrument diagnostic windows.

    Args:
        heights_km: height grid (km).
        log_temp: log10(T[K]) at each height.
        df: instrument table with log_T_min/log_T_max columns.
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.plot(log_temp, heights_km, color='black', lw=2.0)

    color_map = {
        'SOT/NFI': '#1f77b4',
        'SOT/BFI': '#1aa3ff',
        'SOT/SP': '#003c7a',
        'EIS': '#2ca02c',
        'XRT': '#d62728',
    }
    for _, row in df.iterrows():
        color = color_map.get(row['instrument'], 'gray')
        ax.axvspan(
            row['log_T_min'], row['log_T_max'],
            alpha=0.15, color=color,
        )
    # Add labels at the right edge
    label_y_starts = [18000, 16500, 15000, 13500, 12000]
    for (_, row), y_pos in zip(df.iterrows(), label_y_starts):
        color = color_map.get(row['instrument'], 'gray')
        x_mid = 0.5 * (row['log_T_min'] + row['log_T_max'])
        ax.annotate(
            row['instrument'],
            xy=(x_mid, y_pos),
            ha='center', va='center',
            fontweight='bold', color=color,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=color, alpha=0.9),
        )
    # Annotate atmospheric layers
    ax.axhline(500, color='dimgray', linestyle=':', alpha=0.5)
    ax.axhline(2000, color='dimgray', linestyle=':', alpha=0.5)
    ax.axhline(2200, color='dimgray', linestyle=':', alpha=0.5)
    ax.text(7.4, 250, 'photosphere', color='dimgray', fontsize=9)
    ax.text(7.4, 1200, 'chromosphere', color='dimgray', fontsize=9)
    ax.text(7.4, 2100, 'TR', color='dimgray', fontsize=9)
    ax.text(7.4, 8000, 'corona', color='dimgray', fontsize=9)

    ax.set_xlim(3.5, 7.6)
    ax.set_ylim(0, 20000)
    ax.set_xlabel('log T  [K]')
    ax.set_ylabel('height above tau=1  [km]')
    ax.set_title('Hinode multi-layer diagnostic / 다층 진단 — atmosphere + instrument windows')
    plt.tight_layout()
    plt.show()


plot_atmosphere_with_instrument_windows(heights_km, log_temp, instruments)

## Part 4: Simultaneous flare event simulator / 동시 플레어 이벤트 시뮬레이터

We simulate a very simple flare time-evolution and show, on a single time axis, what each Hinode instrument would record as a coordinated observation: SOT detects the photospheric magnetic shear build-up; EIS records the chromospheric/coronal Doppler blueshift of evaporated plasma; XRT sees the soft X-ray flare brightening. This is the **co-temporal / co-spatial value proposition** of the mission.

단순한 플레어 시간 진화 모델을 통해 동일 시간축 위에서 세 기기의 동시 관측을 시뮬레이션한다: SOT는 광구 자기 쉐어 축적, EIS는 증발 플라즈마의 채층/코로나 도플러 청색편이, XRT는 연 X선 밝아짐을 기록. 이것이 임무의 **co-temporal / co-spatial 가치 명제**다.

In [ ]:
def synthesize_flare_signals(t_min: np.ndarray) -> dict:
    """Synthesize idealized flare signals for SOT, EIS, and XRT.

    All quantities are illustrative and have been normalized to unit peaks.

    Args:
        t_min: time array in minutes.

    Returns:
        Dictionary with arrays:
          - sot_shear: photospheric magnetic shear angle (deg) — slow rise.
          - eis_blueshift: EIS Doppler blueshift (km/s) at peak energy release.
          - xrt_flux: soft X-ray flux — fast rise + slow decay (Neupert).
    """
    # Photospheric shear: linear pre-flare build-up, plateau in flare phase
    sot_shear = np.where(
        t_min < 0.0,
        20.0 + 0.5 * t_min,  # build-up before flare onset
        20.0,
    )
    sot_shear = np.where(sot_shear < 0.0, 0.0, sot_shear)
    # EIS blueshift peaks at impulsive phase (Gaussian)
    eis_blueshift = -200.0 * np.exp(-((t_min - 1.0) ** 2) / 1.0)
    # XRT soft X-ray flux: fast rise, slow decay (skewed Gaussian)
    rise_tau = 1.5
    decay_tau = 6.0
    xrt_flux = np.where(
        t_min < 2.0,
        np.exp(-((t_min - 2.0) ** 2) / (2.0 * rise_tau**2)),
        np.exp(-(t_min - 2.0) / decay_tau),
    )
    return {
        'sot_shear_deg': sot_shear,
        'eis_blueshift_km_s': eis_blueshift,
        'xrt_flux_norm': xrt_flux,
    }


t_min = np.linspace(-30.0, 30.0, 600)
signals = synthesize_flare_signals(t_min)

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

axes[0].plot(t_min, signals['sot_shear_deg'], color='#1f77b4', lw=2.0)
axes[0].set_ylabel('SOT shear  [deg]')
axes[0].set_title('SOT — photospheric magnetic shear / 광구 자기 쉐어')
axes[0].axvline(0.0, color='red', linestyle=':', alpha=0.5)

axes[1].plot(t_min, signals['eis_blueshift_km_s'], color='#2ca02c', lw=2.0)
axes[1].set_ylabel('EIS Doppler  [km/s]')
axes[1].set_title('EIS — chromospheric evaporation blueshift / 채층 증발 청색편이')
axes[1].axvline(0.0, color='red', linestyle=':', alpha=0.5)
axes[1].axhline(0.0, color='black', lw=0.5)

axes[2].plot(t_min, signals['xrt_flux_norm'], color='#d62728', lw=2.0)
axes[2].set_ylabel('XRT flux  [norm.]')
axes[2].set_xlabel('time relative to flare onset  [min]')
axes[2].set_title('XRT — soft X-ray flare flux / 연 X선 플레어 광도')
axes[2].axvline(0.0, color='red', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Photospheric vector magnetography / 광구 벡터 자기측정 | SOT/SP 50 cm, 0.16″ pixel, longitudinal 1–5 G | SDO/HMI 4 K full-disk, DKIST 4-m ground |
| EUV imaging spectroscopy / EUV 영상 분광 | EIS 170–290 Å, R≈4000, 3 km/s Doppler | Solar Orbiter/SPICE, IRIS UV |
| Soft X-ray coronal imaging / 연 X선 코로나 영상 | XRT 30 cm Wolter-I, 9 filters, dlogT≈0.2 | SDO/AIA 94/131/335 Å (different optics) |
| Sun-synchronous polar orbit / 태양동기 극궤도 | 680 km, i = 98.1°, 9 months/yr continuous | SDO at GEO (different trade-off) |
| Onboard autonomous flare detection / 탑재 자율 플레어 검출 | MDP flare flag → instrument re-tasking | RHESSI/STIX trigger, IRIS event detector |
| Coordinated three-instrument observatory / 3기기 협조 관측소 | SOT/EIS/XRT shared Z-axis, coordinated obs tables | Solar Orbiter PHI/EUI/SPICE/STIX/EPD |
| Onboard data compression / 탑재 데이터 압축 | DPCM (lossless) + DCT/JPEG (lossy) ASIC | Solar Orbiter, JWST hardware compression |
